In [1]:
import os
from dotenv import load_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis_utils
import isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [2]:
load_dotenv()
#DATA_DIR = os.environ["DATA_DIR"]
# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")
# Define Forecast Months
FC_MONTHS = [9, 10, 11, 12, 1, 2]
FC_PERIOD = slice("2026-09-01", "2027-02-28")

config = analysis_utils.ImpactConfig( version = "v260910",
                                     baseline_period=BASELINE_PERIOD, 
                                     polygons_path= os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
                                     socioeconomics_path= os.environ["POREALLAS_SOCIOECONOMICS_URI"],
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

# Define Forecast
EFFECTS_URI = "/home/emily_zuetell/projects/poreallas/data/v20260909_effects_with_betas.zarr" # Can be any effects datatree

In [4]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [5]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [6]:
# Compute impact: forecast - baseline
impact = config.compute_impact(effect.chunk({dim: -1 for dim in config.dims}), ensemble=True) #Maintain individual ensemble members
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [ ]:
# Aggregate Impact Regions to group_level
group_level = 'ADM1' #IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact.sel(number = [0, 1, 2, 3, 4]), config, group_level)

source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped
source coverage: 6 unmatched source units (first 6: [['BRA.8.823.1961'], ['IDN.14.203'], ['IND.12.129.431'], ['IND.12.129.432'], ['VNM.4.37.365.6048'], ['ZAF.9.313']]); recorded in the manifest and skipped
source coverage: 1 absent_from_data source units (first 1: [['ATA']]); recorded in the manifest and skipped


In [16]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

In [17]:
# Format dataframe for csv output       
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide[stat_col_names] = wide[stat_col_names].round(0).astype("Int64")
wide = wide.reset_index()

In [18]:
wide

,ISO,month 1 median,month 2 median,month 9 median,month 10 median,month 11 median,month 12 median,month 1 p17,month 2 p17,month 9 p17,...,month 9 p10,month 10 p10,month 11 p10,month 12 p10,month 1 p90,month 2 p90,month 9 p90,month 10 p90,month 11 p90,month 12 p90
0,ABW,0,0,0,0,0,0,0,0,-1,...,-1,0,0,0,1,2,0,1,1,1
1,AFG,0,0,2,1,0,0,0,0,-74,...,-117,-27,-1,0,0,1,197,86,15,0
2,AGO,207,232,4,16,117,162,81,102,-8,...,-14,-34,17,36,681,700,86,211,431,574
3,AIA,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ALA,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,WSM,1,2,0,0,0,1,0,1,0,...,0,0,0,0,4,6,0,0,1,2
249,YEM,32,73,1228,288,66,37,5,22,854,...,762,115,12,2,132,222,2413,663,204,143
250,ZAF,378,293,-6,-12,3,59,119,79,-58,...,-134,-133,-46,-15,1343,1129,0,19,183,487
251,ZMB,180,265,-30,25,296,139,56,90,-110,...,-149,-172,-3,-3,498,720,12,271,955,552


In [ ]:
# Output CSV

# Log parameters in filename
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}.csv"

wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
    ),
    index=False,
)

In [20]:
# Groupings for Comms
# SE ASIA
sd_seasia = impact.sel(ISO = ['PHL', 'VNM', 'THA', 'KHM']
                       ).sum(dim = 'ISO').sum(dim = 'month'
                       ).std(dim = config.dims)
se_seasia = sd_seasia/np.sqrt((impact[config.dims[0]].size*impact[config.dims[1]].size))
print(f"SD SE Asia = {sd_seasia.values}")
print(f"SE SE Asia = {se_seasia.values}")
# Sahel
sd_sahel = impact.sel(ISO = ['NGA', 'SDN', 'NER', 'TCD']
                       ).sum(dim = 'ISO').sum(dim = 'month'
                       ).std(dim = config.dims)
se_sahel = sd_sahel/np.sqrt((impact[config.dims[0]].size*impact[config.dims[1]].size))
print(f"SD Sahel = {sd_sahel.values}")
print(f"SE Sahel = {se_sahel.values}")

SD SE Asia = 15665.94352512525
SE SE Asia = 566.4033835111055
SD Sahel = 36980.82296610165
SE Sahel = 1337.0444760911844


In [22]:
(impact[config.dims[0]].size*impact[config.dims[1]].size)

765